In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# ROMANIA - STEP 1
# Rebuild RO 2007/2009 preparation datasets
# - recompute eq_income
# - create age_group
# - keep target model variables
# - output side-by-side diagnostics by year
# =========================================================

base = Path("/content/drive/MyDrive/romania")
files = {
    "RO_2007": base / "RO_2007_master_clean.csv",
    "RO_2009": base / "RO_2009_master_clean.csv",
}

for k, v in files.items():
    print(f"{k}: exists = {v.exists()} | path = {v}")

# 正式模型口径：age 不直接入模，改成 age_group
model_vars_with_y = [
    "HY020", "HX050", "PB040",
    "PB150", "PB220A", "DB040",
    "PE040", "PH010",
    "economic_status", "PL040", "PL050", "HH030", "HS040",
    "age"
]

special_codes = [-1, -2, -3, -4, -5, -6, -7, -8, -9]

num_cols = [
    "HY020", "HX050", "PB040", "age",
    "PE040", "PH010", "economic_status", "PL040", "HH030", "HS040"
]

prepared = {}
summary_rows = []
missing_tables = []
agegroup_tables = []

for dataset_name, path in files.items():
    print("\n" + "=" * 90)
    print(f"PROCESSING {dataset_name}")
    print("=" * 90)

    df = pd.read_csv(path, low_memory=False)

    missing_cols = [c for c in model_vars_with_y if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{dataset_name} missing columns: {missing_cols}")

    work = df[model_vars_with_y].copy()

    # numeric columns -> numeric, special codes -> NaN
    for col in num_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")
        work.loc[work[col].isin(special_codes), col] = np.nan

    # PL050 保持分类变量；特殊码转缺失
    work["PL050"] = work["PL050"].astype("string")
    work.loc[work["PL050"].isin([str(x) for x in special_codes]), "PL050"] = pd.NA

    # recompute eq_income
    work["eq_income"] = np.where(
        work["HY020"].notna() & work["HX050"].notna() & (work["HX050"] > 0),
        work["HY020"] / work["HX050"],
        np.nan
    )

    # create age_group
    work["age_group"] = pd.cut(
        work["age"],
        bins=[0, 14, 24, 54, 64, np.inf],
        labels=["0_14", "15_24", "25_54", "55_64", "65_plus"],
        include_lowest=True,
        right=True
    )

    keep_cols = [
        "eq_income", "PB040",
        "PB150", "PB220A", "DB040",
        "age", "age_group",
        "PE040", "PH010",
        "economic_status", "PL040", "PL050", "HH030", "HS040"
    ]
    work = work[keep_cols].copy()
    prepared[dataset_name] = work

    summary_rows.append({
        "dataset": dataset_name,
        "year": int(dataset_name.split("_")[1]),
        "n_raw": len(df),
        "eq_income_missing": int(work["eq_income"].isna().sum()),
        "eq_income_le_0": int((work["eq_income"] <= 0).sum()),
        "PB040_missing": int(work["PB040"].isna().sum()),
        "PB040_le_0": int((work["PB040"] <= 0).sum()),
        "age_missing": int(work["age"].isna().sum()),
        "age_group_missing": int(work["age_group"].isna().sum())
    })

    miss = work[
        ["eq_income", "PB040", "PB150", "PB220A", "DB040",
         "age_group", "PE040", "PH010",
         "economic_status", "PL040", "PL050", "HH030", "HS040"]
    ].isna().sum().rename(dataset_name)
    missing_tables.append(miss)

    ag = work["age_group"].value_counts(dropna=False).sort_index().rename(dataset_name)
    agegroup_tables.append(ag)

    print("shape after preparation:", work.shape)
    print("eq_income missing:", work["eq_income"].isna().sum())
    print("eq_income <= 0:", (work["eq_income"] <= 0).sum())
    print("age_group counts:")
    print(work["age_group"].value_counts(dropna=False).sort_index())

# combined outputs
summary_df = pd.DataFrame(summary_rows).sort_values("year").reset_index(drop=True)

missing_compare = pd.concat(missing_tables, axis=1)
missing_compare = missing_compare[["RO_2007", "RO_2009"]]

agegroup_compare = pd.concat(agegroup_tables, axis=1)
agegroup_compare = agegroup_compare[["RO_2007", "RO_2009"]]

print("\n" + "=" * 90)
print("COMBINED SUMMARY BY YEAR")
print("=" * 90)
print(summary_df)

print("\n" + "=" * 90)
print("MISSING COUNTS COMPARISON BY YEAR")
print("=" * 90)
print(missing_compare)

print("\n" + "=" * 90)
print("AGE_GROUP DISTRIBUTION BY YEAR")
print("=" * 90)
print(agegroup_compare)

# save outputs
outdir = Path("/content/drive/MyDrive/romania_results")
outdir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(outdir / "RO_2007_2009_step1_summary.csv", index=False)
missing_compare.to_csv(outdir / "RO_2007_2009_step1_missing_compare.csv")
agegroup_compare.to_csv(outdir / "RO_2007_2009_step1_agegroup_compare.csv")

prepared["RO_2007"].to_csv(outdir / "RO_2007_prepared_with_eqincome_agegroup.csv", index=False)
prepared["RO_2009"].to_csv(outdir / "RO_2009_prepared_with_eqincome_agegroup.csv", index=False)

print("\nSaved files:")
print(outdir / "RO_2007_2009_step1_summary.csv")
print(outdir / "RO_2007_2009_step1_missing_compare.csv")
print(outdir / "RO_2007_2009_step1_agegroup_compare.csv")
print(outdir / "RO_2007_prepared_with_eqincome_agegroup.csv")
print(outdir / "RO_2009_prepared_with_eqincome_agegroup.csv")

RO_2007: exists = True | path = /content/drive/MyDrive/romania/RO_2007_master_clean.csv
RO_2009: exists = True | path = /content/drive/MyDrive/romania/RO_2009_master_clean.csv

PROCESSING RO_2007
shape after preparation: (19848, 14)
eq_income missing: 28
eq_income <= 0: 1053
age_group counts:
age_group
0_14           0
15_24       2968
25_54      10241
55_64       2722
65_plus     3917
Name: count, dtype: int64

PROCESSING RO_2009
shape after preparation: (18612, 14)
eq_income missing: 34
eq_income <= 0: 874
age_group counts:
age_group
0_14          0
15_24      2414
25_54      9790
55_64      2824
65_plus    3584
Name: count, dtype: int64

COMBINED SUMMARY BY YEAR
   dataset  year  n_raw  eq_income_missing  eq_income_le_0  PB040_missing  \
0  RO_2007  2007  19848                 28            1053              0   
1  RO_2009  2009  18612                 34             874              0   

   PB040_le_0  age_missing  age_group_missing  
0           0            0                  0 

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 2
# Build ONE common complete-case sample per year
# based on FULL model covariates
# Then both extended model and full model will use the SAME rows
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = Path("/content/drive/MyDrive/romania_results")
output_dir.mkdir(parents=True, exist_ok=True)

files = {
    "RO_2007": input_dir / "RO_2007_prepared_with_eqincome_agegroup.csv",
    "RO_2009": input_dir / "RO_2009_prepared_with_eqincome_agegroup.csv",
}

# age 不再进模型，改成 age_group
# 共同样本按 full model 所需全部变量删缺失
full_covariates_with_y = [
    "eq_income", "PB040",
    "PB150", "PB220A", "DB040",
    "age_group", "PE040", "PH010",
    "economic_status", "PL040", "PL050", "HH030", "HS040"
]

all_summary = []
all_cc = []

for dataset_name, path in files.items():
    print("\n" + "=" * 90)
    print(f"BUILDING COMMON COMPLETE-CASE SAMPLE: {dataset_name}")
    print("=" * 90)

    df = pd.read_csv(path, low_memory=False)

    d = df[full_covariates_with_y].copy()

    n_prepared = len(d)
    print("n_prepared:", n_prepared)

    # 先筛 eq_income 和 PB040 有效
    d = d[
        d["eq_income"].notna() &
        (d["eq_income"] > 0) &
        d["PB040"].notna() &
        (d["PB040"] > 0)
    ].copy()

    n_after_income_weight = len(d)
    print("n_after_valid_eq_income_and_weight:", n_after_income_weight)

    # full complete-case：任何 covariate 缺失就删
    d_cc = d.dropna().copy()

    n_common_cc = len(d_cc)
    print("n_common_complete_case:", n_common_cc)
    print("rows_dropped_at_complete_case_step:", n_after_income_weight - n_common_cc)

    print("\nMissing counts AFTER common complete-case:")
    print(d_cc.isna().sum())

    # 年份标签
    d_cc["dataset"] = dataset_name
    d_cc["year"] = int(dataset_name.split("_")[1])

    save_path = output_dir / f"{dataset_name}_common_cc_for_extended_and_full.csv"
    d_cc.to_csv(save_path, index=False)

    print("\nSaved common sample to:")
    print(save_path)

    all_cc.append(d_cc)

    all_summary.append(pd.DataFrame({
        "dataset": [dataset_name],
        "year": [int(dataset_name.split("_")[1])],
        "n_prepared": [n_prepared],
        "n_after_valid_eq_income_and_weight": [n_after_income_weight],
        "n_common_complete_case": [n_common_cc],
        "rows_dropped_at_complete_case_step": [n_after_income_weight - n_common_cc]
    }))

summary_df = pd.concat(all_summary, ignore_index=True).sort_values("year").reset_index(drop=True)
combined_cc = pd.concat(all_cc, ignore_index=True).sort_values("year").reset_index(drop=True)

print("\n" + "=" * 90)
print("COMMON SAMPLE SUMMARY BY YEAR")
print("=" * 90)
print(summary_df)

summary_path = output_dir / "RO_2007_2009_common_cc_summary.csv"
combined_path = output_dir / "RO_2007_2009_common_cc_combined.csv"

summary_df.to_csv(summary_path, index=False)
combined_cc.to_csv(combined_path, index=False)

print("\nSaved summary to:")
print(summary_path)

print("\nSaved combined common sample to:")
print(combined_path)


BUILDING COMMON COMPLETE-CASE SAMPLE: RO_2007
n_prepared: 19848
n_after_valid_eq_income_and_weight: 18767
n_common_complete_case: 13648
rows_dropped_at_complete_case_step: 5119

Missing counts AFTER common complete-case:
eq_income          0
PB040              0
PB150              0
PB220A             0
DB040              0
age_group          0
PE040              0
PH010              0
economic_status    0
PL040              0
PL050              0
HH030              0
HS040              0
dtype: int64

Saved common sample to:
/content/drive/MyDrive/romania_results/RO_2007_common_cc_for_extended_and_full.csv

BUILDING COMMON COMPLETE-CASE SAMPLE: RO_2009
n_prepared: 18612
n_after_valid_eq_income_and_weight: 17704
n_common_complete_case: 12977
rows_dropped_at_complete_case_step: 4727

Missing counts AFTER common complete-case:
eq_income          0
PB040              0
PB150              0
PB220A             0
DB040              0
age_group          0
PE040              0
PH010          

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 2
# Build ONE common complete-case sample per year
# based on FULL model covariates
# Then both extended model and full model will use the SAME rows
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = Path("/content/drive/MyDrive/romania_results")
output_dir.mkdir(parents=True, exist_ok=True)

files = {
    "RO_2007": input_dir / "RO_2007_prepared_with_eqincome_agegroup.csv",
    "RO_2009": input_dir / "RO_2009_prepared_with_eqincome_agegroup.csv",
}

full_covariates_with_y = [
    "eq_income", "PB040",
    "PB150", "PB220A", "DB040",
    "age_group", "PE040", "PH010",
    "economic_status", "PL040", "PL050", "HH030", "HS040"
]

all_summary = []
all_cc = []

for dataset_name, path in files.items():
    print("\n" + "=" * 90)
    print(f"BUILDING COMMON COMPLETE-CASE SAMPLE: {dataset_name}")
    print("=" * 90)

    df = pd.read_csv(path, low_memory=False)
    d = df[full_covariates_with_y].copy()

    n_prepared = len(d)
    print("n_prepared:", n_prepared)

    d = d[
        d["eq_income"].notna() &
        (d["eq_income"] > 0) &
        d["PB040"].notna() &
        (d["PB040"] > 0)
    ].copy()

    n_after_income_weight = len(d)
    print("n_after_valid_eq_income_and_weight:", n_after_income_weight)

    d_cc = d.dropna().copy()

    n_common_cc = len(d_cc)
    print("n_common_complete_case:", n_common_cc)
    print("rows_dropped_at_complete_case_step:", n_after_income_weight - n_common_cc)

    print("\nMissing counts AFTER common complete-case:")
    print(d_cc.isna().sum())

    d_cc["dataset"] = dataset_name
    d_cc["year"] = int(dataset_name.split("_")[1])

    save_path = output_dir / f"{dataset_name}_common_cc_for_extended_and_full.csv"
    d_cc.to_csv(save_path, index=False)

    print("\nSaved common sample to:")
    print(save_path)

    all_cc.append(d_cc)

    all_summary.append(pd.DataFrame({
        "dataset": [dataset_name],
        "year": [int(dataset_name.split("_")[1])],
        "n_prepared": [n_prepared],
        "n_after_valid_eq_income_and_weight": [n_after_income_weight],
        "n_common_complete_case": [n_common_cc],
        "rows_dropped_at_complete_case_step": [n_after_income_weight - n_common_cc]
    }))

summary_df = pd.concat(all_summary, ignore_index=True).sort_values("year").reset_index(drop=True)
combined_cc = pd.concat(all_cc, ignore_index=True).sort_values("year").reset_index(drop=True)

print("\n" + "=" * 90)
print("COMMON SAMPLE SUMMARY BY YEAR")
print("=" * 90)
print(summary_df)

summary_path = output_dir / "RO_2007_2009_common_cc_summary.csv"
combined_path = output_dir / "RO_2007_2009_common_cc_combined.csv"

summary_df.to_csv(summary_path, index=False)
combined_cc.to_csv(combined_path, index=False)

print("\nSaved summary to:")
print(summary_path)

print("\nSaved combined common sample to:")
print(combined_path)


BUILDING COMMON COMPLETE-CASE SAMPLE: RO_2007
n_prepared: 19848
n_after_valid_eq_income_and_weight: 18767
n_common_complete_case: 13648
rows_dropped_at_complete_case_step: 5119

Missing counts AFTER common complete-case:
eq_income          0
PB040              0
PB150              0
PB220A             0
DB040              0
age_group          0
PE040              0
PH010              0
economic_status    0
PL040              0
PL050              0
HH030              0
HS040              0
dtype: int64

Saved common sample to:
/content/drive/MyDrive/romania_results/RO_2007_common_cc_for_extended_and_full.csv

BUILDING COMMON COMPLETE-CASE SAMPLE: RO_2009
n_prepared: 18612
n_after_valid_eq_income_and_weight: 17704
n_common_complete_case: 12977
rows_dropped_at_complete_case_step: 4727

Missing counts AFTER common complete-case:
eq_income          0
PB040              0
PB150              0
PB220A             0
DB040              0
age_group          0
PE040              0
PH010          

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 3A
# Inspect observed levels and common levels across 2007/2009
# for all categorical covariates
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")

files = {
    "RO_2007": input_dir / "RO_2007_common_cc_for_extended_and_full.csv",
    "RO_2009": input_dir / "RO_2009_common_cc_for_extended_and_full.csv",
}

covariates = [
    "PB150", "PB220A", "DB040", "age_group", "PE040", "PH010",
    "economic_status", "PL040", "PL050", "HH030", "HS040"
]

d2007 = pd.read_csv(files["RO_2007"], low_memory=False)
d2009 = pd.read_csv(files["RO_2009"], low_memory=False)

for col in covariates:
    d2007[col] = d2007[col].astype("string")
    d2009[col] = d2009[col].astype("string")

rows = []

print("=" * 110)
print("ROMANIA STEP 3A - OBSERVED LEVELS BY YEAR")
print("=" * 110)

for var in covariates:
    levels_2007 = sorted(d2007[var].dropna().unique().tolist())
    levels_2009 = sorted(d2009[var].dropna().unique().tolist())
    common_levels = sorted(set(levels_2007).intersection(set(levels_2009)))

    print(f"\n--- {var} ---")
    print("2007 levels:")
    print(levels_2007)
    print("2009 levels:")
    print(levels_2009)
    print("Common levels in both years:")
    print(common_levels)

    rows.append({
        "variable": var,
        "n_levels_2007": len(levels_2007),
        "n_levels_2009": len(levels_2009),
        "n_common_levels": len(common_levels),
        "levels_2007": " | ".join(levels_2007),
        "levels_2009": " | ".join(levels_2009),
        "common_levels": " | ".join(common_levels)
    })

summary = pd.DataFrame(rows)

out_path = input_dir / "RO_2007_2009_observed_levels_summary.csv"
summary.to_csv(out_path, index=False)

print("\n" + "=" * 110)
print("SUMMARY TABLE")
print("=" * 110)
print(summary[["variable", "n_levels_2007", "n_levels_2009", "n_common_levels"]])

print("\nSaved to:")
print(out_path)

ROMANIA STEP 3A - OBSERVED LEVELS BY YEAR

--- PB150 ---
2007 levels:
['1', '2']
2009 levels:
['1', '2']
Common levels in both years:
['1', '2']

--- PB220A ---
2007 levels:
['EU', 'Other', 'RO']
2009 levels:
['EU', 'Other', 'RO']
Common levels in both years:
['EU', 'Other', 'RO']

--- DB040 ---
2007 levels:
['RO01', 'RO02', 'RO03', 'RO04', 'RO05', 'RO06', 'RO07', 'RO08']
2009 levels:
['RO11', 'RO12', 'RO21', 'RO22', 'RO31', 'RO32', 'RO41', 'RO42']
Common levels in both years:
[]

--- age_group ---
2007 levels:
['15_24', '25_54', '55_64', '65_plus']
2009 levels:
['15_24', '25_54', '55_64', '65_plus']
Common levels in both years:
['15_24', '25_54', '55_64', '65_plus']

--- PE040 ---
2007 levels:
['1.0', '2.0', '3.0', '4.0', '5.0', '6.0']
2009 levels:
['1.0', '2.0', '3.0', '4.0', '5.0', '6.0']
Common levels in both years:
['1.0', '2.0', '3.0', '4.0', '5.0', '6.0']

--- PH010 ---
2007 levels:
['1.0', '2.0', '3.0', '4.0', '5.0']
2009 levels:
['1.0', '2.0', '3.0', '4.0', '5.0']
Common level

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 3B (REVISED)
# Harmonize DB040 across 2007/2009 to common region labels,
# then validate one shared reference map
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")

files = {
    "RO_2007": input_dir / "RO_2007_common_cc_for_extended_and_full.csv",
    "RO_2009": input_dir / "RO_2009_common_cc_for_extended_and_full.csv",
}

# ---------------------------------------------------------
# 1. DB040 harmonization map
# Use one common label system across years
# ---------------------------------------------------------
db040_map = {
    "RO_2007": {
        "RO01": "Nord_Est",
        "RO02": "Sud_Est",
        "RO03": "Sud_Muntenia",
        "RO04": "Sud_Vest_Oltenia",
        "RO05": "Vest",
        "RO06": "Nord_Vest",
        "RO07": "Centru",
        "RO08": "Bucuresti_Ilfov",
    },
    "RO_2009": {
        "RO11": "Nord_Vest",
        "RO12": "Centru",
        "RO21": "Nord_Est",
        "RO22": "Sud_Est",
        "RO31": "Sud_Muntenia",
        "RO32": "Bucuresti_Ilfov",
        "RO41": "Sud_Vest_Oltenia",
        "RO42": "Vest",
    }
}

# ---------------------------------------------------------
# 2. Shared reference map AFTER DB040 harmonization
# ---------------------------------------------------------
reference_map = {
    "PB150": "1",
    "PB220A": "RO",
    "DB040_region": "Nord_Est",
    "age_group": "25_54",
    "PE040": "1.0",
    "PH010": "2.0",
    "economic_status": "1.0",
    "PL040": "3.0",
    "PL050": "9",
    "HH030": "6.0",
    "HS040": "1.0"
}

covariates_for_reference = [
    "PB150", "PB220A", "DB040_region", "age_group", "PE040", "PH010",
    "economic_status", "PL040", "PL050", "HH030", "HS040"
]

harmonized_dfs = {}
rows = []

print("=" * 110)
print("ROMANIA STEP 3B (REVISED) - DB040 HARMONIZATION + REFERENCE VALIDATION")
print("=" * 110)

for dataset_name, path in files.items():
    print(f"\nDATASET: {dataset_name}")
    df = pd.read_csv(path, low_memory=False)

    # convert relevant existing categorical columns to string
    original_cat_cols = [
        "PB150", "PB220A", "DB040", "age_group", "PE040", "PH010",
        "economic_status", "PL040", "PL050", "HH030", "HS040"
    ]
    for col in original_cat_cols:
        df[col] = df[col].astype("string")

    # harmonize DB040
    df["DB040_region"] = df["DB040"].map(db040_map[dataset_name]).astype("string")

    unmapped_raw = sorted(df.loc[df["DB040_region"].isna(), "DB040"].dropna().unique().tolist())
    print("Unmapped DB040 raw codes:", unmapped_raw)

    print("Observed DB040 raw levels:")
    print(sorted(df["DB040"].dropna().unique().tolist()))

    print("Observed DB040_region harmonized levels:")
    print(sorted(df["DB040_region"].dropna().unique().tolist()))

    # validate reference presence
    for var in covariates_for_reference:
        ref = reference_map[var]
        levels = sorted(df[var].dropna().unique().tolist())
        ref_found = ref in levels

        rows.append({
            "dataset": dataset_name,
            "variable": var,
            "reference": ref,
            "reference_found": ref_found,
            "n_levels": len(levels),
            "all_levels": " | ".join(levels)
        })

        print(f"{var}: reference = {ref} | found = {ref_found} | n_levels = {len(levels)}")

    harmonized_dfs[dataset_name] = df

# ---------------------------------------------------------
# 3. Cross-year check for harmonized DB040_region
# ---------------------------------------------------------
levels_2007 = sorted(harmonized_dfs["RO_2007"]["DB040_region"].dropna().unique().tolist())
levels_2009 = sorted(harmonized_dfs["RO_2009"]["DB040_region"].dropna().unique().tolist())
common_levels = sorted(set(levels_2007).intersection(set(levels_2009)))

print("\n" + "=" * 110)
print("CROSS-YEAR CHECK FOR DB040_region")
print("=" * 110)
print("RO_2007 DB040_region levels:")
print(levels_2007)
print("RO_2009 DB040_region levels:")
print(levels_2009)
print("Common levels:")
print(common_levels)

reference_summary = pd.DataFrame(rows)
invalid = reference_summary[~reference_summary["reference_found"]].copy()

print("\n" + "=" * 110)
print("REFERENCE SUMMARY")
print("=" * 110)
print(reference_summary[["dataset", "variable", "reference", "reference_found", "n_levels"]])

print("\n" + "=" * 110)
print("INVALID REFERENCES (should be empty)")
print("=" * 110)
print(invalid)

# ---------------------------------------------------------
# 4. Save outputs
# ---------------------------------------------------------
for dataset_name, df in harmonized_dfs.items():
    save_path = input_dir / f"{dataset_name}_common_cc_harmonized_DB040.csv"
    df.to_csv(save_path, index=False)
    print(f"\nSaved harmonized dataset: {save_path}")

summary_path = input_dir / "RO_2007_2009_reference_summary_harmonized_DB040.csv"
reference_summary.to_csv(summary_path, index=False)

print("\nSaved reference summary:")
print(summary_path)

ROMANIA STEP 3B (REVISED) - DB040 HARMONIZATION + REFERENCE VALIDATION

DATASET: RO_2007
Unmapped DB040 raw codes: []
Observed DB040 raw levels:
['RO01', 'RO02', 'RO03', 'RO04', 'RO05', 'RO06', 'RO07', 'RO08']
Observed DB040_region harmonized levels:
['Bucuresti_Ilfov', 'Centru', 'Nord_Est', 'Nord_Vest', 'Sud_Est', 'Sud_Muntenia', 'Sud_Vest_Oltenia', 'Vest']
PB150: reference = 1 | found = True | n_levels = 2
PB220A: reference = RO | found = True | n_levels = 3
DB040_region: reference = Nord_Est | found = True | n_levels = 8
age_group: reference = 25_54 | found = True | n_levels = 4
PE040: reference = 1.0 | found = True | n_levels = 6
PH010: reference = 2.0 | found = True | n_levels = 5
economic_status: reference = 1.0 | found = True | n_levels = 8
PL040: reference = 3.0 | found = True | n_levels = 4
PL050: reference = 9 | found = True | n_levels = 9
HH030: reference = 6.0 | found = True | n_levels = 6
HS040: reference = 1.0 | found = True | n_levels = 2

DATASET: RO_2009
Unmapped DB040

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 4
# Create model-ready datasets with fixed reference categories
# - keep original DB040_raw for audit
# - use harmonized region labels as modeling DB040
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = Path("/content/drive/MyDrive/romania_results")
output_dir.mkdir(parents=True, exist_ok=True)

files = {
    "RO_2007": input_dir / "RO_2007_common_cc_harmonized_DB040.csv",
    "RO_2009": input_dir / "RO_2009_common_cc_harmonized_DB040.csv",
}

# Final chosen references for modeling
reference_map = {
    "PB150": "1",
    "PB220A": "RO",
    "DB040": "Nord_Est",   # harmonized region label
    "age_group": "25_54",
    "PE040": "1.0",
    "PH010": "2.0",
    "economic_status": "1.0",
    "PL040": "3.0",
    "PL050": "9",
    "HH030": "6.0",
    "HS040": "1.0"
}

categorical_vars = list(reference_map.keys())

def reorder_with_reference(series, ref):
    s = series.astype("string")
    levels = sorted([x for x in s.dropna().unique().tolist()])
    if ref not in levels:
        raise ValueError(f"Reference {ref} not found in {series.name}")
    new_levels = [ref] + [x for x in levels if x != ref]
    return pd.Categorical(s, categories=new_levels, ordered=False), new_levels

reference_check_rows = []

for dataset_name, path in files.items():
    print("\n" + "=" * 100)
    print(f"PREPARING MODEL-READY DATASET: {dataset_name}")
    print("=" * 100)

    df = pd.read_csv(path, low_memory=False)

    # ------------------------------------------------------
    # Keep original raw DB040 code for audit,
    # but replace modeling DB040 with harmonized labels
    # ------------------------------------------------------
    df["DB040_raw"] = df["DB040"].astype("string")
    df["DB040"] = df["DB040_region"].astype("string")

    # Optional: keep DB040_region too for transparency
    df["DB040_region"] = df["DB040_region"].astype("string")

    level_report = {}

    for var in categorical_vars:
        df[var], levels = reorder_with_reference(df[var], reference_map[var])
        level_report[var] = levels
        reference_check_rows.append({
            "dataset": dataset_name,
            "variable": var,
            "reference": reference_map[var],
            "first_level_after_reorder": levels[0],
            "n_levels": len(levels),
            "all_levels": " | ".join(levels)
        })

    # Put key modeling columns first
    front_cols = [
        "eq_income", "PB040",
        "PB150", "PB220A", "DB040", "age_group",
        "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
    ]
    other_cols = [c for c in df.columns if c not in front_cols]
    df = df[front_cols + other_cols].copy()

    save_path = output_dir / f"{dataset_name}_model_ready_with_references.csv"
    df.to_csv(save_path, index=False)

    print("Saved to:")
    print(save_path)

    print("\nFirst level check (should always equal chosen reference):")
    for var in categorical_vars:
        print(f"{var}: {level_report[var][0]}")

    print("\nModeling DB040 levels:")
    print(level_report["DB040"])

reference_check_df = pd.DataFrame(reference_check_rows)

print("\n" + "=" * 100)
print("REFERENCE ORDER CHECK SUMMARY")
print("=" * 100)
print(reference_check_df[[
    "dataset", "variable", "reference",
    "first_level_after_reorder", "n_levels"
]])

summary_path = output_dir / "RO_2007_2009_model_ready_reference_order_check.csv"
reference_check_df.to_csv(summary_path, index=False)

print("\nSaved summary to:")
print(summary_path)


PREPARING MODEL-READY DATASET: RO_2007
Saved to:
/content/drive/MyDrive/romania_results/RO_2007_model_ready_with_references.csv

First level check (should always equal chosen reference):
PB150: 1
PB220A: RO
DB040: Nord_Est
age_group: 25_54
PE040: 1.0
PH010: 2.0
economic_status: 1.0
PL040: 3.0
PL050: 9
HH030: 6.0
HS040: 1.0

Modeling DB040 levels:
['Nord_Est', 'Bucuresti_Ilfov', 'Centru', 'Nord_Vest', 'Sud_Est', 'Sud_Muntenia', 'Sud_Vest_Oltenia', 'Vest']

PREPARING MODEL-READY DATASET: RO_2009
Saved to:
/content/drive/MyDrive/romania_results/RO_2009_model_ready_with_references.csv

First level check (should always equal chosen reference):
PB150: 1
PB220A: RO
DB040: Nord_Est
age_group: 25_54
PE040: 1.0
PH010: 2.0
economic_status: 1.0
PL040: 3.0
PL050: 9
HH030: 6.0
HS040: 1.0

Modeling DB040 levels:
['Nord_Est', 'Bucuresti_Ilfov', 'Centru', 'Nord_Vest', 'Sud_Est', 'Sud_Muntenia', 'Sud_Vest_Oltenia', 'Vest']

REFERENCE ORDER CHECK SUMMARY
    dataset         variable reference first_leve

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
# =========================================================
# ROMANIA - STEP 5A
# RO_2007 only
# Re-apply references INSIDE R
# Run unpenalized LR for extended and full on the SAME sample
# =========================================================

if (!requireNamespace("LorenzRegression", quietly = TRUE)) {
  install.packages("LorenzRegression", repos = "https://cloud.r-project.org")
}
library(LorenzRegression)

# ---------------------------------------------------------
# 1. read data
# ---------------------------------------------------------
d <- read.csv(
  "/content/drive/MyDrive/romania_results/RO_2007_model_ready_with_references.csv",
  stringsAsFactors = FALSE
)

cat("=== Raw input data ===\n")
print(dim(d))

# keep only modeling columns
d <- d[, c(
  "eq_income", "PB040",
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)]

cat("\n=== Columns kept for modeling ===\n")
print(names(d))
cat("n =", nrow(d), "\n")

# ---------------------------------------------------------
# 2. helper: normalize categorical codes
# ---------------------------------------------------------
normalize_code <- function(x) {
  y <- as.character(x)
  y <- trimws(y)
  y <- sub("\\.0$", "", y)
  y
}

cat_vars <- c(
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)

for (v in cat_vars) {
  d[[v]] <- normalize_code(d[[v]])
}

d$eq_income <- as.numeric(d$eq_income)
d$PB040 <- as.numeric(d$PB040)

# ---------------------------------------------------------
# 3. helper: put chosen reference first, keep all observed levels
# ---------------------------------------------------------
levels_with_ref_first <- function(x, ref) {
  lev <- sort(unique(x[!is.na(x)]))
  if (!(ref %in% lev)) {
    stop(paste("Reference", ref, "not found in", deparse(substitute(x))))
  }
  c(ref, lev[lev != ref])
}

# ---------------------------------------------------------
# 4. explicit factor levels / references
# ---------------------------------------------------------
d$PB150 <- factor(d$PB150, levels = levels_with_ref_first(d$PB150, "1"))
d$PB220A <- factor(d$PB220A, levels = levels_with_ref_first(d$PB220A, "RO"))
d$DB040 <- factor(d$DB040, levels = levels_with_ref_first(d$DB040, "Nord_Est"))
d$age_group <- factor(d$age_group, levels = levels_with_ref_first(d$age_group, "25_54"))
d$PE040 <- factor(d$PE040, levels = levels_with_ref_first(d$PE040, "1"))
d$PH010 <- factor(d$PH010, levels = levels_with_ref_first(d$PH010, "2"))
d$economic_status <- factor(d$economic_status, levels = levels_with_ref_first(d$economic_status, "1"))
d$PL040 <- factor(d$PL040, levels = levels_with_ref_first(d$PL040, "3"))
d$PL050 <- factor(d$PL050, levels = levels_with_ref_first(d$PL050, "9"))
d$HH030 <- factor(d$HH030, levels = levels_with_ref_first(d$HH030, "6"))
d$HS040 <- factor(d$HS040, levels = levels_with_ref_first(d$HS040, "1"))

# ---------------------------------------------------------
# 5. check reference + number of observed levels
# ---------------------------------------------------------
cat("\n=== Reference check ===\n")
for (v in cat_vars) {
  cat(v, "reference =", levels(d[[v]])[1], "\n")
}

cat("\n=== Number of observed non-empty levels after factor conversion ===\n")
level_check <- sapply(cat_vars, function(v) nlevels(droplevels(d[[v]])))
print(level_check)

bad_vars <- names(level_check[level_check < 2])
if (length(bad_vars) > 0) {
  cat("\nVariables with <2 observed levels:\n")
  print(bad_vars)
  stop("At least one factor has fewer than 2 observed levels. Stop here.")
}

# ---------------------------------------------------------
# 6. final missing check
# ---------------------------------------------------------
cat("\n=== Missing counts just before modeling ===\n")
print(colSums(is.na(d)))

# ---------------------------------------------------------
# 7. weights
# ---------------------------------------------------------
d$w_norm <- d$PB040 / sum(d$PB040)

cat("\n=== Weight check ===\n")
print(summary(d$w_norm))
print(sum(d$w_norm))

# ---------------------------------------------------------
# 8. overall Gini
# ---------------------------------------------------------
overall_gini <- Gini.coef(y = d$eq_income, weights = d$w_norm)

cat("\n=== Overall Gini ===\n")
print(overall_gini)

# ---------------------------------------------------------
# 9. extended model
# ---------------------------------------------------------
set.seed(123)
fit_ext <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010,
  data = d,
  weights = d$w_norm,
  penalty = "none",
  na.action = na.omit
)

cat("\n=== EXTENDED unpenalized LR ===\n")
print(fit_ext)

cat("\nExtended theta:\n")
print(fit_ext$theta)

cat("\nExtended explained Gini:\n")
print(fit_ext$Gi.expl)

cat("\nExtended LR2:\n")
print(fit_ext$LR2)

# ---------------------------------------------------------
# 10. full model
# ---------------------------------------------------------
set.seed(123)
fit_full <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010 +
    economic_status + PL040 + PL050 + HH030 + HS040,
  data = d,
  weights = d$w_norm,
  penalty = "none",
  na.action = na.omit
)

cat("\n=== FULL unpenalized LR ===\n")
print(fit_full)

cat("\nFull theta:\n")
print(fit_full$theta)

cat("\nFull explained Gini:\n")
print(fit_full$Gi.expl)

cat("\nFull LR2:\n")
print(fit_full$LR2)

# ---------------------------------------------------------
# 11. save outputs
# ---------------------------------------------------------
metrics <- data.frame(
  dataset = "RO_2007",
  model = c("extended_unpenalized", "full_unpenalized"),
  n_used = c(nrow(d), nrow(d)),
  Gini_total = c(as.numeric(overall_gini), as.numeric(overall_gini)),
  Gi_expl = c(as.numeric(fit_ext$Gi.expl), as.numeric(fit_full$Gi.expl)),
  LR2 = c(as.numeric(fit_ext$LR2), as.numeric(fit_full$LR2))
)

theta_ext <- data.frame(
  dataset = "RO_2007",
  model = "extended_unpenalized",
  term = names(fit_ext$theta),
  theta = as.numeric(fit_ext$theta)
)

theta_full <- data.frame(
  dataset = "RO_2007",
  model = "full_unpenalized",
  term = names(fit_full$theta),
  theta = as.numeric(fit_full$theta)
)

theta_all <- rbind(theta_ext, theta_full)

write.csv(
  metrics,
  "/content/drive/MyDrive/romania_results/RO_2007_unpenalized_same_sample_metrics.csv",
  row.names = FALSE
)

write.csv(
  theta_all,
  "/content/drive/MyDrive/romania_results/RO_2007_unpenalized_same_sample_theta.csv",
  row.names = FALSE
)

cat("\n=== Saved metrics ===\n")
print(metrics)
cat("\nSaved to: /content/drive/MyDrive/romania_results/RO_2007_unpenalized_same_sample_metrics.csv\n")
cat("Saved to: /content/drive/MyDrive/romania_results/RO_2007_unpenalized_same_sample_theta.csv\n")

=== Raw input data ===
[1] 13648    17

=== Columns kept for modeling ===
 [1] "eq_income"       "PB040"           "PB150"           "PB220A"         
 [5] "DB040"           "age_group"       "PE040"           "PH010"          
 [9] "economic_status" "PL040"           "PL050"           "HH030"          
[13] "HS040"          
n = 13648 

=== Reference check ===
PB150 reference = 1 
PB220A reference = RO 
DB040 reference = Nord_Est 
age_group reference = 25_54 
PE040 reference = 1 
PH010 reference = 2 
economic_status reference = 1 
PL040 reference = 3 
PL050 reference = 9 
HH030 reference = 6 
HS040 reference = 1 

=== Number of observed non-empty levels after factor conversion ===
          PB150          PB220A           DB040       age_group           PE040 
              2               3               8               4               6 
          PH010 economic_status           PL040           PL050           HH030 
              5               8               4               9   

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘warp’, ‘SparseM’, ‘MatrixModels’, ‘globals’, ‘hardhat’, ‘sparsevctrs’, ‘furrr’, ‘slider’, ‘iterators’, ‘quantreg’, ‘parsnip’, ‘rsample’, ‘doParallel’, ‘foreach’, ‘GA’, ‘Rearrangement’, ‘RcppArmadillo’

trying URL 'https://cloud.r-project.org/src/contrib/listenv_0.10.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/parallelly_1.46.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/future_1.70.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/warp_0.2.3.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/SparseM_1.84-2.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/MatrixModels_0.5-4.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/globals_0.19.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/hardhat_1.4.3.tar.gz'
trying URL 'https://cloud.r-project.org/src/co

In [ ]:
%%R
# =========================================================
# ROMANIA - STEP 5B
# RO_2009 only
# Re-apply references INSIDE R
# Run unpenalized LR for extended and full on the SAME sample
# =========================================================

library(LorenzRegression)

# ---------------------------------------------------------
# 1. read data
# ---------------------------------------------------------
d <- read.csv(
  "/content/drive/MyDrive/romania_results/RO_2009_model_ready_with_references.csv",
  stringsAsFactors = FALSE
)

cat("=== Raw input data ===\n")
print(dim(d))

# keep only modeling columns
d <- d[, c(
  "eq_income", "PB040",
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)]

cat("\n=== Columns kept for modeling ===\n")
print(names(d))
cat("n =", nrow(d), "\n")

# ---------------------------------------------------------
# 2. helper: normalize categorical codes
# ---------------------------------------------------------
normalize_code <- function(x) {
  y <- as.character(x)
  y <- trimws(y)
  y <- sub("\\.0$", "", y)
  y
}

cat_vars <- c(
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)

for (v in cat_vars) {
  d[[v]] <- normalize_code(d[[v]])
}

d$eq_income <- as.numeric(d$eq_income)
d$PB040 <- as.numeric(d$PB040)

# ---------------------------------------------------------
# 3. helper: put chosen reference first, keep all observed levels
# ---------------------------------------------------------
levels_with_ref_first <- function(x, ref) {
  lev <- sort(unique(x[!is.na(x)]))
  if (!(ref %in% lev)) {
    stop(paste("Reference", ref, "not found in", deparse(substitute(x))))
  }
  c(ref, lev[lev != ref])
}

# ---------------------------------------------------------
# 4. explicit factor levels / references
# ---------------------------------------------------------
d$PB150 <- factor(d$PB150, levels = levels_with_ref_first(d$PB150, "1"))
d$PB220A <- factor(d$PB220A, levels = levels_with_ref_first(d$PB220A, "RO"))
d$DB040 <- factor(d$DB040, levels = levels_with_ref_first(d$DB040, "Nord_Est"))
d$age_group <- factor(d$age_group, levels = levels_with_ref_first(d$age_group, "25_54"))
d$PE040 <- factor(d$PE040, levels = levels_with_ref_first(d$PE040, "1"))
d$PH010 <- factor(d$PH010, levels = levels_with_ref_first(d$PH010, "2"))
d$economic_status <- factor(d$economic_status, levels = levels_with_ref_first(d$economic_status, "1"))
d$PL040 <- factor(d$PL040, levels = levels_with_ref_first(d$PL040, "3"))
d$PL050 <- factor(d$PL050, levels = levels_with_ref_first(d$PL050, "9"))
d$HH030 <- factor(d$HH030, levels = levels_with_ref_first(d$HH030, "6"))
d$HS040 <- factor(d$HS040, levels = levels_with_ref_first(d$HS040, "1"))

# ---------------------------------------------------------
# 5. check reference + number of observed levels
# ---------------------------------------------------------
cat("\n=== Reference check ===\n")
for (v in cat_vars) {
  cat(v, "reference =", levels(d[[v]])[1], "\n")
}

cat("\n=== Number of observed non-empty levels after factor conversion ===\n")
level_check <- sapply(cat_vars, function(v) nlevels(droplevels(d[[v]])))
print(level_check)

bad_vars <- names(level_check[level_check < 2])
if (length(bad_vars) > 0) {
  cat("\nVariables with <2 observed levels:\n")
  print(bad_vars)
  stop("At least one factor has fewer than 2 observed levels. Stop here.")
}

# ---------------------------------------------------------
# 6. final missing check
# ---------------------------------------------------------
cat("\n=== Missing counts just before modeling ===\n")
print(colSums(is.na(d)))

# ---------------------------------------------------------
# 7. weights
# ---------------------------------------------------------
d$w_norm <- d$PB040 / sum(d$PB040)

cat("\n=== Weight check ===\n")
print(summary(d$w_norm))
print(sum(d$w_norm))

# ---------------------------------------------------------
# 8. overall Gini
# ---------------------------------------------------------
overall_gini <- Gini.coef(y = d$eq_income, weights = d$w_norm)

cat("\n=== Overall Gini ===\n")
print(overall_gini)

# ---------------------------------------------------------
# 9. extended model
# ---------------------------------------------------------
set.seed(123)
fit_ext <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010,
  data = d,
  weights = d$w_norm,
  penalty = "none",
  na.action = na.omit
)

cat("\n=== EXTENDED unpenalized LR ===\n")
print(fit_ext)

cat("\nExtended theta:\n")
print(fit_ext$theta)

cat("\nExtended explained Gini:\n")
print(fit_ext$Gi.expl)

cat("\nExtended LR2:\n")
print(fit_ext$LR2)

# ---------------------------------------------------------
# 10. full model
# ---------------------------------------------------------
set.seed(123)
fit_full <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010 +
    economic_status + PL040 + PL050 + HH030 + HS040,
  data = d,
  weights = d$w_norm,
  penalty = "none",
  na.action = na.omit
)

cat("\n=== FULL unpenalized LR ===\n")
print(fit_full)

cat("\nFull theta:\n")
print(fit_full$theta)

cat("\nFull explained Gini:\n")
print(fit_full$Gi.expl)

cat("\nFull LR2:\n")
print(fit_full$LR2)

# ---------------------------------------------------------
# 11. save outputs
# ---------------------------------------------------------
metrics <- data.frame(
  dataset = "RO_2009",
  model = c("extended_unpenalized", "full_unpenalized"),
  n_used = c(nrow(d), nrow(d)),
  Gini_total = c(as.numeric(overall_gini), as.numeric(overall_gini)),
  Gi_expl = c(as.numeric(fit_ext$Gi.expl), as.numeric(fit_full$Gi.expl)),
  LR2 = c(as.numeric(fit_ext$LR2), as.numeric(fit_full$LR2))
)

theta_ext <- data.frame(
  dataset = "RO_2009",
  model = "extended_unpenalized",
  term = names(fit_ext$theta),
  theta = as.numeric(fit_ext$theta)
)

theta_full <- data.frame(
  dataset = "RO_2009",
  model = "full_unpenalized",
  term = names(fit_full$theta),
  theta = as.numeric(fit_full$theta)
)

theta_all <- rbind(theta_ext, theta_full)

write.csv(
  metrics,
  "/content/drive/MyDrive/romania_results/RO_2009_unpenalized_same_sample_metrics.csv",
  row.names = FALSE
)

write.csv(
  theta_all,
  "/content/drive/MyDrive/romania_results/RO_2009_unpenalized_same_sample_theta.csv",
  row.names = FALSE
)

cat("\n=== Saved metrics ===\n")
print(metrics)
cat("\nSaved to: /content/drive/MyDrive/romania_results/RO_2009_unpenalized_same_sample_metrics.csv\n")
cat("Saved to: /content/drive/MyDrive/romania_results/RO_2009_unpenalized_same_sample_theta.csv\n")

=== Raw input data ===
[1] 12977    17

=== Columns kept for modeling ===
 [1] "eq_income"       "PB040"           "PB150"           "PB220A"         
 [5] "DB040"           "age_group"       "PE040"           "PH010"          
 [9] "economic_status" "PL040"           "PL050"           "HH030"          
[13] "HS040"          
n = 12977 

=== Reference check ===
PB150 reference = 1 
PB220A reference = RO 
DB040 reference = Nord_Est 
age_group reference = 25_54 
PE040 reference = 1 
PH010 reference = 2 
economic_status reference = 1 
PL040 reference = 3 
PL050 reference = 9 
HH030 reference = 6 
HS040 reference = 1 

=== Number of observed non-empty levels after factor conversion ===
          PB150          PB220A           DB040       age_group           PE040 
              2               3               8               4               6 
          PH010 economic_status           PL040           PL050           HH030 
              5              10               4               9   

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 5C
# Combine 2007 and 2009 unpenalized same-sample metrics
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = input_dir

f_2007 = input_dir / "RO_2007_unpenalized_same_sample_metrics.csv"
f_2009 = input_dir / "RO_2009_unpenalized_same_sample_metrics.csv"

m2007 = pd.read_csv(f_2007)
m2009 = pd.read_csv(f_2009)

combined = pd.concat([m2007, m2009], ignore_index=True)

# keep a clean order
model_order = {
    "extended_unpenalized": 1,
    "full_unpenalized": 2
}
year_order = {
    "RO_2007": 2007,
    "RO_2009": 2009
}

combined["year"] = combined["dataset"].map(year_order)
combined["model_order"] = combined["model"].map(model_order)

combined = combined.sort_values(
    ["year", "model_order"]
).reset_index(drop=True)

combined = combined[
    ["dataset", "year", "model", "n_used", "Gini_total", "Gi_expl", "LR2"]
].copy()

print("=" * 100)
print("ROMANIA UNPENALIZED SAME-SAMPLE METRICS COMBINED")
print("=" * 100)
print(combined)

wide = combined.pivot(
    index="year",
    columns="model",
    values=["n_used", "Gini_total", "Gi_expl", "LR2"]
)

wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

print("\n" + "=" * 100)
print("WIDE COMPARISON TABLE")
print("=" * 100)
print(wide)

save_long = output_dir / "RO_2007_2009_unpenalized_same_sample_metrics_combined.csv"
save_wide = output_dir / "RO_2007_2009_unpenalized_same_sample_metrics_wide.csv"

combined.to_csv(save_long, index=False)
wide.to_csv(save_wide, index=False)

print("\nSaved to:")
print(save_long)
print(save_wide)

ROMANIA UNPENALIZED SAME-SAMPLE METRICS COMBINED
   dataset  year                 model  n_used  Gini_total   Gi_expl       LR2
0  RO_2007  2007  extended_unpenalized   13648    0.398588  0.132356  0.332062
1  RO_2007  2007      full_unpenalized   13648    0.398588  0.145086  0.364001
2  RO_2009  2009  extended_unpenalized   12977    0.360920  0.116595  0.323049
3  RO_2009  2009      full_unpenalized   12977    0.360920  0.139160  0.385569

WIDE COMPARISON TABLE
   year  n_used_extended_unpenalized  n_used_full_unpenalized  \
0  2007                      13648.0                  13648.0   
1  2009                      12977.0                  12977.0   

   Gini_total_extended_unpenalized  Gini_total_full_unpenalized  \
0                         0.398588                     0.398588   
1                         0.360920                     0.360920   

   Gi_expl_extended_unpenalized  Gi_expl_full_unpenalized  \
0                      0.132356                  0.145086   
1            

In [ ]:
%%R
# =========================================================
# ROMANIA - STEP 6A
# RO_2007 only
# SCAD-BIC penalized Lorenz regression
# same common sample, same references
# =========================================================

if (!requireNamespace("LorenzRegression", quietly = TRUE)) {
  install.packages("LorenzRegression", repos = "https://cloud.r-project.org")
}
library(LorenzRegression)

# ---------------------------------------------------------
# 1. read data
# ---------------------------------------------------------
d <- read.csv(
  "/content/drive/MyDrive/romania_results/RO_2007_model_ready_with_references.csv",
  stringsAsFactors = FALSE
)

cat("=== Raw input data ===\n")
print(dim(d))

# keep only modeling columns
d <- d[, c(
  "eq_income", "PB040",
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)]

cat("\n=== Columns kept for modeling ===\n")
print(names(d))
cat("n =", nrow(d), "\n")

# ---------------------------------------------------------
# 2. helper: normalize categorical codes
# ---------------------------------------------------------
normalize_code <- function(x) {
  y <- as.character(x)
  y <- trimws(y)
  y <- sub("\\.0$", "", y)
  y
}

cat_vars <- c(
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)

for (v in cat_vars) {
  d[[v]] <- normalize_code(d[[v]])
}

d$eq_income <- as.numeric(d$eq_income)
d$PB040 <- as.numeric(d$PB040)

# ---------------------------------------------------------
# 3. helper: put chosen reference first, keep all observed levels
# ---------------------------------------------------------
levels_with_ref_first <- function(x, ref) {
  lev <- sort(unique(x[!is.na(x)]))
  if (!(ref %in% lev)) {
    stop(paste("Reference", ref, "not found in", deparse(substitute(x))))
  }
  c(ref, lev[lev != ref])
}

# ---------------------------------------------------------
# 4. explicit factor levels / references
# ---------------------------------------------------------
d$PB150 <- factor(d$PB150, levels = levels_with_ref_first(d$PB150, "1"))
d$PB220A <- factor(d$PB220A, levels = levels_with_ref_first(d$PB220A, "RO"))
d$DB040 <- factor(d$DB040, levels = levels_with_ref_first(d$DB040, "Nord_Est"))
d$age_group <- factor(d$age_group, levels = levels_with_ref_first(d$age_group, "25_54"))
d$PE040 <- factor(d$PE040, levels = levels_with_ref_first(d$PE040, "1"))
d$PH010 <- factor(d$PH010, levels = levels_with_ref_first(d$PH010, "2"))
d$economic_status <- factor(d$economic_status, levels = levels_with_ref_first(d$economic_status, "1"))
d$PL040 <- factor(d$PL040, levels = levels_with_ref_first(d$PL040, "3"))
d$PL050 <- factor(d$PL050, levels = levels_with_ref_first(d$PL050, "9"))
d$HH030 <- factor(d$HH030, levels = levels_with_ref_first(d$HH030, "6"))
d$HS040 <- factor(d$HS040, levels = levels_with_ref_first(d$HS040, "1"))

# ---------------------------------------------------------
# 5. diagnostics before modeling
# ---------------------------------------------------------
cat("\n=== Reference check ===\n")
for (v in cat_vars) {
  cat(v, "reference =", levels(d[[v]])[1], "\n")
}

cat("\n=== Missing counts just before modeling ===\n")
print(colSums(is.na(d)))

cat("\n=== Number of observed non-empty levels ===\n")
level_check <- sapply(cat_vars, function(v) nlevels(droplevels(d[[v]])))
print(level_check)

bad_vars <- names(level_check[level_check < 2])
if (length(bad_vars) > 0) {
  cat("\nVariables with <2 observed levels:\n")
  print(bad_vars)
  stop("At least one factor has fewer than 2 observed levels. Stop here.")
}

# ---------------------------------------------------------
# 6. weights + overall Gini
# ---------------------------------------------------------
d$w_norm <- d$PB040 / sum(d$PB040)

cat("\n=== Weight check ===\n")
print(summary(d$w_norm))
print(sum(d$w_norm))

overall_gini <- Gini.coef(y = d$eq_income, weights = d$w_norm)

cat("\n=== Overall Gini ===\n")
print(overall_gini)

# ---------------------------------------------------------
# 7. EXTENDED SCAD-BIC
# ---------------------------------------------------------
set.seed(123)
fit_ext_scad <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010,
  data = d,
  weights = d$w_norm,
  penalty = "SCAD",
  na.action = na.omit
)

cat("\n=== EXTENDED SCAD fit ===\n")
print(fit_ext_scad)

cat("\n=== summary(EXTENDED SCAD fit) ===\n")
print(summary(fit_ext_scad))

coef_ext_bic <- coef(fit_ext_scad, pars.idx = "BIC", renormalize = TRUE)
gini_ext_bic <- ineqExplained(fit_ext_scad, type = "Gini.explained", pars.idx = "BIC")
lr2_ext_bic  <- ineqExplained(fit_ext_scad, type = "Lorenz-R2", pars.idx = "BIC")

cat("\n=== EXTENDED BIC-selected coefficients ===\n")
print(coef_ext_bic)

cat("\n=== EXTENDED BIC-selected explained Gini ===\n")
print(gini_ext_bic)

cat("\n=== EXTENDED BIC-selected Lorenz-R2 ===\n")
print(lr2_ext_bic)

nonzero_ext <- abs(coef_ext_bic) > 1e-10

cat("\n=== EXTENDED number selected ===\n")
print(sum(nonzero_ext))

cat("\n=== EXTENDED selected nonzero terms ===\n")
print(coef_ext_bic[nonzero_ext])

# ---------------------------------------------------------
# 8. FULL SCAD-BIC
# ---------------------------------------------------------
set.seed(123)
fit_full_scad <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010 +
    economic_status + PL040 + PL050 + HH030 + HS040,
  data = d,
  weights = d$w_norm,
  penalty = "SCAD",
  na.action = na.omit
)

cat("\n=== FULL SCAD fit ===\n")
print(fit_full_scad)

cat("\n=== summary(FULL SCAD fit) ===\n")
print(summary(fit_full_scad))

coef_full_bic <- coef(fit_full_scad, pars.idx = "BIC", renormalize = TRUE)
gini_full_bic <- ineqExplained(fit_full_scad, type = "Gini.explained", pars.idx = "BIC")
lr2_full_bic  <- ineqExplained(fit_full_scad, type = "Lorenz-R2", pars.idx = "BIC")

cat("\n=== FULL BIC-selected coefficients ===\n")
print(coef_full_bic)

cat("\n=== FULL BIC-selected explained Gini ===\n")
print(gini_full_bic)

cat("\n=== FULL BIC-selected Lorenz-R2 ===\n")
print(lr2_full_bic)

nonzero_full <- abs(coef_full_bic) > 1e-10

cat("\n=== FULL number selected ===\n")
print(sum(nonzero_full))

cat("\n=== FULL selected nonzero terms ===\n")
print(coef_full_bic[nonzero_full])

# ---------------------------------------------------------
# 9. save outputs
# ---------------------------------------------------------
metrics <- data.frame(
  dataset = "RO_2007",
  model = c("extended_scad_bic", "full_scad_bic"),
  n_used = c(nrow(d), nrow(d)),
  Gini_total = c(as.numeric(overall_gini), as.numeric(overall_gini)),
  Gi_expl = c(as.numeric(gini_ext_bic), as.numeric(gini_full_bic)),
  LR2 = c(as.numeric(lr2_ext_bic), as.numeric(lr2_full_bic)),
  n_selected = c(sum(nonzero_ext), sum(nonzero_full))
)

theta_ext <- data.frame(
  dataset = "RO_2007",
  model = "extended_scad_bic",
  term = names(coef_ext_bic),
  theta = as.numeric(coef_ext_bic)
)

theta_full <- data.frame(
  dataset = "RO_2007",
  model = "full_scad_bic",
  term = names(coef_full_bic),
  theta = as.numeric(coef_full_bic)
)

theta_all <- rbind(theta_ext, theta_full)

write.csv(
  metrics,
  "/content/drive/MyDrive/romania_results/RO_2007_scad_bic_same_sample_metrics.csv",
  row.names = FALSE
)

write.csv(
  theta_all,
  "/content/drive/MyDrive/romania_results/RO_2007_scad_bic_same_sample_theta.csv",
  row.names = FALSE
)

cat("\n=== Saved metrics ===\n")
print(metrics)
cat("\nSaved to: /content/drive/MyDrive/romania_results/RO_2007_scad_bic_same_sample_metrics.csv\n")
cat("Saved to: /content/drive/MyDrive/romania_results/RO_2007_scad_bic_same_sample_theta.csv\n")

=== Raw input data ===
[1] 13648    17

=== Columns kept for modeling ===
 [1] "eq_income"       "PB040"           "PB150"           "PB220A"         
 [5] "DB040"           "age_group"       "PE040"           "PH010"          
 [9] "economic_status" "PL040"           "PL050"           "HH030"          
[13] "HS040"          
n = 13648 

=== Reference check ===
PB150 reference = 1 
PB220A reference = RO 
DB040 reference = Nord_Est 
age_group reference = 25_54 
PE040 reference = 1 
PH010 reference = 2 
economic_status reference = 1 
PL040 reference = 3 
PL050 reference = 9 
HH030 reference = 6 
HS040 reference = 1 

=== Missing counts just before modeling ===
      eq_income           PB040           PB150          PB220A           DB040 
              0               0               0               0               0 
      age_group           PE040           PH010 economic_status           PL040 
              0               0               0               0               0 
         

In [ ]:
%%R
# =========================================================
# ROMANIA - STEP 6B
# RO_2009 only
# SCAD-BIC penalized Lorenz regression
# same common sample, same references
# =========================================================

if (!requireNamespace("LorenzRegression", quietly = TRUE)) {
  install.packages("LorenzRegression", repos = "https://cloud.r-project.org")
}
library(LorenzRegression)

# ---------------------------------------------------------
# 1. read data
# ---------------------------------------------------------
d <- read.csv(
  "/content/drive/MyDrive/romania_results/RO_2009_model_ready_with_references.csv",
  stringsAsFactors = FALSE
)

cat("=== Raw input data ===\n")
print(dim(d))

# keep only modeling columns
d <- d[, c(
  "eq_income", "PB040",
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)]

cat("\n=== Columns kept for modeling ===\n")
print(names(d))
cat("n =", nrow(d), "\n")

# ---------------------------------------------------------
# 2. helper: normalize categorical codes
# ---------------------------------------------------------
normalize_code <- function(x) {
  y <- as.character(x)
  y <- trimws(y)
  y <- sub("\\.0$", "", y)
  y
}

cat_vars <- c(
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)

for (v in cat_vars) {
  d[[v]] <- normalize_code(d[[v]])
}

d$eq_income <- as.numeric(d$eq_income)
d$PB040 <- as.numeric(d$PB040)

# ---------------------------------------------------------
# 3. helper: put chosen reference first, keep all observed levels
# ---------------------------------------------------------
levels_with_ref_first <- function(x, ref) {
  lev <- sort(unique(x[!is.na(x)]))
  if (!(ref %in% lev)) {
    stop(paste("Reference", ref, "not found in", deparse(substitute(x))))
  }
  c(ref, lev[lev != ref])
}

# ---------------------------------------------------------
# 4. explicit factor levels / references
# ---------------------------------------------------------
d$PB150 <- factor(d$PB150, levels = levels_with_ref_first(d$PB150, "1"))
d$PB220A <- factor(d$PB220A, levels = levels_with_ref_first(d$PB220A, "RO"))
d$DB040 <- factor(d$DB040, levels = levels_with_ref_first(d$DB040, "Nord_Est"))
d$age_group <- factor(d$age_group, levels = levels_with_ref_first(d$age_group, "25_54"))
d$PE040 <- factor(d$PE040, levels = levels_with_ref_first(d$PE040, "1"))
d$PH010 <- factor(d$PH010, levels = levels_with_ref_first(d$PH010, "2"))
d$economic_status <- factor(d$economic_status, levels = levels_with_ref_first(d$economic_status, "1"))
d$PL040 <- factor(d$PL040, levels = levels_with_ref_first(d$PL040, "3"))
d$PL050 <- factor(d$PL050, levels = levels_with_ref_first(d$PL050, "9"))
d$HH030 <- factor(d$HH030, levels = levels_with_ref_first(d$HH030, "6"))
d$HS040 <- factor(d$HS040, levels = levels_with_ref_first(d$HS040, "1"))

# ---------------------------------------------------------
# 5. diagnostics before modeling
# ---------------------------------------------------------
cat("\n=== Reference check ===\n")
for (v in cat_vars) {
  cat(v, "reference =", levels(d[[v]])[1], "\n")
}

cat("\n=== Missing counts just before modeling ===\n")
print(colSums(is.na(d)))

cat("\n=== Number of observed non-empty levels ===\n")
level_check <- sapply(cat_vars, function(v) nlevels(droplevels(d[[v]])))
print(level_check)

bad_vars <- names(level_check[level_check < 2])
if (length(bad_vars) > 0) {
  cat("\nVariables with <2 observed levels:\n")
  print(bad_vars)
  stop("At least one factor has fewer than 2 observed levels. Stop here.")
}

# ---------------------------------------------------------
# 6. weights + overall Gini
# ---------------------------------------------------------
d$w_norm <- d$PB040 / sum(d$PB040)

cat("\n=== Weight check ===\n")
print(summary(d$w_norm))
print(sum(d$w_norm))

overall_gini <- Gini.coef(y = d$eq_income, weights = d$w_norm)

cat("\n=== Overall Gini ===\n")
print(overall_gini)

# ---------------------------------------------------------
# 7. EXTENDED SCAD-BIC
# ---------------------------------------------------------
set.seed(123)
fit_ext_scad <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010,
  data = d,
  weights = d$w_norm,
  penalty = "SCAD",
  na.action = na.omit
)

cat("\n=== EXTENDED SCAD fit ===\n")
print(fit_ext_scad)

cat("\n=== summary(EXTENDED SCAD fit) ===\n")
print(summary(fit_ext_scad))

coef_ext_bic <- coef(fit_ext_scad, pars.idx = "BIC", renormalize = TRUE)
gini_ext_bic <- ineqExplained(fit_ext_scad, type = "Gini.explained", pars.idx = "BIC")
lr2_ext_bic  <- ineqExplained(fit_ext_scad, type = "Lorenz-R2", pars.idx = "BIC")

cat("\n=== EXTENDED BIC-selected coefficients ===\n")
print(coef_ext_bic)

cat("\n=== EXTENDED BIC-selected explained Gini ===\n")
print(gini_ext_bic)

cat("\n=== EXTENDED BIC-selected Lorenz-R2 ===\n")
print(lr2_ext_bic)

nonzero_ext <- abs(coef_ext_bic) > 1e-10

cat("\n=== EXTENDED number selected ===\n")
print(sum(nonzero_ext))

cat("\n=== EXTENDED selected nonzero terms ===\n")
print(coef_ext_bic[nonzero_ext])

# ---------------------------------------------------------
# 8. FULL SCAD-BIC
# ---------------------------------------------------------
set.seed(123)
fit_full_scad <- Lorenz.Reg(
  eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010 +
    economic_status + PL040 + PL050 + HH030 + HS040,
  data = d,
  weights = d$w_norm,
  penalty = "SCAD",
  na.action = na.omit
)

cat("\n=== FULL SCAD fit ===\n")
print(fit_full_scad)

cat("\n=== summary(FULL SCAD fit) ===\n")
print(summary(fit_full_scad))

coef_full_bic <- coef(fit_full_scad, pars.idx = "BIC", renormalize = TRUE)
gini_full_bic <- ineqExplained(fit_full_scad, type = "Gini.explained", pars.idx = "BIC")
lr2_full_bic  <- ineqExplained(fit_full_scad, type = "Lorenz-R2", pars.idx = "BIC")

cat("\n=== FULL BIC-selected coefficients ===\n")
print(coef_full_bic)

cat("\n=== FULL BIC-selected explained Gini ===\n")
print(gini_full_bic)

cat("\n=== FULL BIC-selected Lorenz-R2 ===\n")
print(lr2_full_bic)

nonzero_full <- abs(coef_full_bic) > 1e-10

cat("\n=== FULL number selected ===\n")
print(sum(nonzero_full))

cat("\n=== FULL selected nonzero terms ===\n")
print(coef_full_bic[nonzero_full])

# ---------------------------------------------------------
# 9. save outputs
# ---------------------------------------------------------
metrics <- data.frame(
  dataset = "RO_2009",
  model = c("extended_scad_bic", "full_scad_bic"),
  n_used = c(nrow(d), nrow(d)),
  Gini_total = c(as.numeric(overall_gini), as.numeric(overall_gini)),
  Gi_expl = c(as.numeric(gini_ext_bic), as.numeric(gini_full_bic)),
  LR2 = c(as.numeric(lr2_ext_bic), as.numeric(lr2_full_bic)),
  n_selected = c(sum(nonzero_ext), sum(nonzero_full))
)

theta_ext <- data.frame(
  dataset = "RO_2009",
  model = "extended_scad_bic",
  term = names(coef_ext_bic),
  theta = as.numeric(coef_ext_bic)
)

theta_full <- data.frame(
  dataset = "RO_2009",
  model = "full_scad_bic",
  term = names(coef_full_bic),
  theta = as.numeric(coef_full_bic)
)

theta_all <- rbind(theta_ext, theta_full)

write.csv(
  metrics,
  "/content/drive/MyDrive/romania_results/RO_2009_scad_bic_same_sample_metrics.csv",
  row.names = FALSE
)

write.csv(
  theta_all,
  "/content/drive/MyDrive/romania_results/RO_2009_scad_bic_same_sample_theta.csv",
  row.names = FALSE
)

cat("\n=== Saved metrics ===\n")
print(metrics)
cat("\nSaved to: /content/drive/MyDrive/romania_results/RO_2009_scad_bic_same_sample_metrics.csv\n")
cat("Saved to: /content/drive/MyDrive/romania_results/RO_2009_scad_bic_same_sample_theta.csv\n")

=== Raw input data ===
[1] 12977    17

=== Columns kept for modeling ===
 [1] "eq_income"       "PB040"           "PB150"           "PB220A"         
 [5] "DB040"           "age_group"       "PE040"           "PH010"          
 [9] "economic_status" "PL040"           "PL050"           "HH030"          
[13] "HS040"          
n = 12977 

=== Reference check ===
PB150 reference = 1 
PB220A reference = RO 
DB040 reference = Nord_Est 
age_group reference = 25_54 
PE040 reference = 1 
PH010 reference = 2 
economic_status reference = 1 
PL040 reference = 3 
PL050 reference = 9 
HH030 reference = 6 
HS040 reference = 1 

=== Missing counts just before modeling ===
      eq_income           PB040           PB150          PB220A           DB040 
              0               0               0               0               0 
      age_group           PE040           PH010 economic_status           PL040 
              0               0               0               0               0 
         

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - STEP 6C
# Combine 2007 and 2009 SCAD-BIC same-sample metrics
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = input_dir

f_2007 = input_dir / "RO_2007_scad_bic_same_sample_metrics.csv"
f_2009 = input_dir / "RO_2009_scad_bic_same_sample_metrics.csv"

print("RO_2007 file exists:", f_2007.exists(), "|", f_2007)
print("RO_2009 file exists:", f_2009.exists(), "|", f_2009)

m2007 = pd.read_csv(f_2007)
m2009 = pd.read_csv(f_2009)

combined = pd.concat([m2007, m2009], ignore_index=True)

model_order = {
    "extended_scad_bic": 1,
    "full_scad_bic": 2
}
year_order = {
    "RO_2007": 2007,
    "RO_2009": 2009
}

combined["year"] = combined["dataset"].map(year_order)
combined["model_order"] = combined["model"].map(model_order)

combined = combined.sort_values(
    ["year", "model_order"]
).reset_index(drop=True)

combined = combined[
    ["dataset", "year", "model", "n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
].copy()

print("\n" + "=" * 100)
print("ROMANIA SCAD-BIC SAME-SAMPLE METRICS COMBINED")
print("=" * 100)
print(combined)

wide = combined.pivot(
    index="year",
    columns="model",
    values=["n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
)

wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

print("\n" + "=" * 100)
print("WIDE SCAD-BIC COMPARISON TABLE")
print("=" * 100)
print(wide)

save_long = output_dir / "RO_2007_2009_scad_bic_same_sample_metrics_combined.csv"
save_wide = output_dir / "RO_2007_2009_scad_bic_same_sample_metrics_wide.csv"

combined.to_csv(save_long, index=False)
wide.to_csv(save_wide, index=False)

print("\nSaved to:")
print(save_long)
print(save_wide)

RO_2007 file exists: True | /content/drive/MyDrive/romania_results/RO_2007_scad_bic_same_sample_metrics.csv
RO_2009 file exists: True | /content/drive/MyDrive/romania_results/RO_2009_scad_bic_same_sample_metrics.csv

ROMANIA SCAD-BIC SAME-SAMPLE METRICS COMBINED
   dataset  year              model  n_used  Gini_total   Gi_expl       LR2  \
0  RO_2007  2007  extended_scad_bic   13648    0.398588  0.138480  0.334935   
1  RO_2007  2007      full_scad_bic   13648    0.398588  0.165407  0.400062   
2  RO_2009  2009  extended_scad_bic   12977    0.360920  0.102861  0.295500   
3  RO_2009  2009      full_scad_bic   12977    0.360920  0.129571  0.372233   

   n_selected  
0          21  
1          38  
2          16  
3          41  

WIDE SCAD-BIC COMPARISON TABLE
   year  n_used_extended_scad_bic  n_used_full_scad_bic  \
0  2007                   13648.0               13648.0   
1  2009                   12977.0               12977.0   

   Gini_total_extended_scad_bic  Gini_total_full_sc

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - FINAL STEP
# Combine unpenalized + SCAD-BIC into one final summary table
# and create within-year improvement tables
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = input_dir

# ---------------------------------------------------------
# 1. read the two combined metric tables
# ---------------------------------------------------------
f_unpen = input_dir / "RO_2007_2009_unpenalized_same_sample_metrics_combined.csv"
f_scad  = input_dir / "RO_2007_2009_scad_bic_same_sample_metrics_combined.csv"

print("unpenalized file exists:", f_unpen.exists(), "|", f_unpen)
print("scad-bic file exists:", f_scad.exists(), "|", f_scad)

u = pd.read_csv(f_unpen)
s = pd.read_csv(f_scad)

# ---------------------------------------------------------
# 2. harmonize columns
# ---------------------------------------------------------
if "n_selected" not in u.columns:
    u["n_selected"] = pd.NA

final_long = pd.concat([u, s], ignore_index=True)

year_order = {2007: 1, 2009: 2}
model_order = {
    "extended_unpenalized": 1,
    "extended_scad_bic": 2,
    "full_unpenalized": 3,
    "full_scad_bic": 4,
}

final_long["year_order"] = final_long["year"].map(year_order)
final_long["model_order"] = final_long["model"].map(model_order)

final_long = final_long.sort_values(
    ["year_order", "model_order"]
).reset_index(drop=True)

final_long = final_long[
    ["dataset", "year", "model", "n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
].copy()

print("\n" + "=" * 110)
print("ROMANIA FINAL LONG SUMMARY TABLE")
print("=" * 110)
print(final_long)

# ---------------------------------------------------------
# 3. create wide summary table
# ---------------------------------------------------------
final_wide = final_long.pivot(
    index="year",
    columns="model",
    values=["n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
)

final_wide.columns = [f"{a}_{b}" for a, b in final_wide.columns]
final_wide = final_wide.reset_index()

print("\n" + "=" * 110)
print("ROMANIA FINAL WIDE SUMMARY TABLE")
print("=" * 110)
print(final_wide)

# ---------------------------------------------------------
# 4. within-year improvement tables:
#    penalized minus unpenalized
# ---------------------------------------------------------
pivot_for_delta = final_long.pivot(
    index="year",
    columns="model",
    values=["Gi_expl", "LR2"]
)

delta = pd.DataFrame({
    "year": pivot_for_delta.index,
    "delta_Gi_expl_extended_scad_minus_unpen": (
        pivot_for_delta[("Gi_expl", "extended_scad_bic")] -
        pivot_for_delta[("Gi_expl", "extended_unpenalized")]
    ).values,
    "delta_LR2_extended_scad_minus_unpen": (
        pivot_for_delta[("LR2", "extended_scad_bic")] -
        pivot_for_delta[("LR2", "extended_unpenalized")]
    ).values,
    "delta_Gi_expl_full_scad_minus_unpen": (
        pivot_for_delta[("Gi_expl", "full_scad_bic")] -
        pivot_for_delta[("Gi_expl", "full_unpenalized")]
    ).values,
    "delta_LR2_full_scad_minus_unpen": (
        pivot_for_delta[("LR2", "full_scad_bic")] -
        pivot_for_delta[("LR2", "full_unpenalized")]
    ).values,
})

print("\n" + "=" * 110)
print("ROMANIA WITHIN-YEAR PENALIZED MINUS UNPENALIZED DELTA TABLE")
print("=" * 110)
print(delta)

# ---------------------------------------------------------
# 5. save outputs
# ---------------------------------------------------------
save_long = output_dir / "RO_final_summary_long.csv"
save_wide = output_dir / "RO_final_summary_wide.csv"
save_delta = output_dir / "RO_final_penalized_minus_unpenalized_delta.csv"

final_long.to_csv(save_long, index=False)
final_wide.to_csv(save_wide, index=False)
delta.to_csv(save_delta, index=False)

print("\nSaved to:")
print(save_long)
print(save_wide)
print(save_delta)

unpenalized file exists: True | /content/drive/MyDrive/romania_results/RO_2007_2009_unpenalized_same_sample_metrics_combined.csv
scad-bic file exists: True | /content/drive/MyDrive/romania_results/RO_2007_2009_scad_bic_same_sample_metrics_combined.csv

ROMANIA FINAL LONG SUMMARY TABLE
   dataset  year                 model  n_used  Gini_total   Gi_expl  \
0  RO_2007  2007  extended_unpenalized   13648    0.398588  0.132356   
1  RO_2007  2007     extended_scad_bic   13648    0.398588  0.138480   
2  RO_2007  2007      full_unpenalized   13648    0.398588  0.145086   
3  RO_2007  2007         full_scad_bic   13648    0.398588  0.165407   
4  RO_2009  2009  extended_unpenalized   12977    0.360920  0.116595   
5  RO_2009  2009     extended_scad_bic   12977    0.360920  0.102861   
6  RO_2009  2009      full_unpenalized   12977    0.360920  0.139160   
7  RO_2009  2009         full_scad_bic   12977    0.360920  0.129571   

        LR2 n_selected  
0  0.332062       <NA>  
1  0.334935    

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - FIX PENALIZED SUMMARY CONSISTENCY
# Keep Gini_total = external overall_gini
# Recompute penalized LR2 as Gi_expl / Gini_total
# Then rebuild corrected final summary tables
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = input_dir

# ---------------------------------------------------------
# 1. read source tables
# ---------------------------------------------------------
f_unpen = input_dir / "RO_2007_2009_unpenalized_same_sample_metrics_combined.csv"
f_scad  = input_dir / "RO_2007_2009_scad_bic_same_sample_metrics_combined.csv"

u = pd.read_csv(f_unpen)
s = pd.read_csv(f_scad)

print("Loaded:")
print(f_unpen)
print(f_scad)

# ---------------------------------------------------------
# 2. audit: keep original package LR2, then recompute consistent LR2
# ---------------------------------------------------------
s = s.copy()
s["LR2_pkg_original"] = s["LR2"]
s["LR2_recomputed_from_Gi_over_Gini"] = s["Gi_expl"] / s["Gini_total"]
s["LR2"] = s["LR2_recomputed_from_Gi_over_Gini"]

print("\n" + "=" * 110)
print("AUDIT TABLE: ORIGINAL VS RECOMPUTED SCAD-BIC LR2")
print("=" * 110)
audit_cols = [
    "dataset", "year", "model", "Gini_total", "Gi_expl",
    "LR2_pkg_original", "LR2_recomputed_from_Gi_over_Gini"
]
print(s[audit_cols])

# ---------------------------------------------------------
# 3. save corrected SCAD table
# ---------------------------------------------------------
scad_corrected_long = s[[
    "dataset", "year", "model", "n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"
]].copy()

scad_corrected_wide = scad_corrected_long.pivot(
    index="year",
    columns="model",
    values=["n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
)
scad_corrected_wide.columns = [f"{a}_{b}" for a, b in scad_corrected_wide.columns]
scad_corrected_wide = scad_corrected_wide.reset_index()

save_scad_long = output_dir / "RO_2007_2009_scad_bic_same_sample_metrics_combined_corrected.csv"
save_scad_wide = output_dir / "RO_2007_2009_scad_bic_same_sample_metrics_wide_corrected.csv"
save_scad_audit = output_dir / "RO_2007_2009_scad_bic_LR2_audit.csv"

scad_corrected_long.to_csv(save_scad_long, index=False)
scad_corrected_wide.to_csv(save_scad_wide, index=False)
s[audit_cols].to_csv(save_scad_audit, index=False)

# ---------------------------------------------------------
# 4. rebuild final long summary
# ---------------------------------------------------------
if "n_selected" not in u.columns:
    u["n_selected"] = pd.NA

final_long = pd.concat([u, scad_corrected_long], ignore_index=True)

year_order = {2007: 1, 2009: 2}
model_order = {
    "extended_unpenalized": 1,
    "extended_scad_bic": 2,
    "full_unpenalized": 3,
    "full_scad_bic": 4,
}

final_long["year_order"] = final_long["year"].map(year_order)
final_long["model_order"] = final_long["model"].map(model_order)

final_long = final_long.sort_values(
    ["year_order", "model_order"]
).reset_index(drop=True)

final_long = final_long[
    ["dataset", "year", "model", "n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
].copy()

print("\n" + "=" * 110)
print("CORRECTED ROMANIA FINAL LONG SUMMARY TABLE")
print("=" * 110)
print(final_long)

# ---------------------------------------------------------
# 5. rebuild final wide summary
# ---------------------------------------------------------
final_wide = final_long.pivot(
    index="year",
    columns="model",
    values=["n_used", "Gini_total", "Gi_expl", "LR2", "n_selected"]
)

final_wide.columns = [f"{a}_{b}" for a, b in final_wide.columns]
final_wide = final_wide.reset_index()

print("\n" + "=" * 110)
print("CORRECTED ROMANIA FINAL WIDE SUMMARY TABLE")
print("=" * 110)
print(final_wide)

# ---------------------------------------------------------
# 6. corrected within-year delta table
# ---------------------------------------------------------
pivot_for_delta = final_long.pivot(
    index="year",
    columns="model",
    values=["Gi_expl", "LR2"]
)

delta = pd.DataFrame({
    "year": pivot_for_delta.index,
    "delta_Gi_expl_extended_scad_minus_unpen": (
        pivot_for_delta[("Gi_expl", "extended_scad_bic")] -
        pivot_for_delta[("Gi_expl", "extended_unpenalized")]
    ).values,
    "delta_LR2_extended_scad_minus_unpen": (
        pivot_for_delta[("LR2", "extended_scad_bic")] -
        pivot_for_delta[("LR2", "extended_unpenalized")]
    ).values,
    "delta_Gi_expl_full_scad_minus_unpen": (
        pivot_for_delta[("Gi_expl", "full_scad_bic")] -
        pivot_for_delta[("Gi_expl", "full_unpenalized")]
    ).values,
    "delta_LR2_full_scad_minus_unpen": (
        pivot_for_delta[("LR2", "full_scad_bic")] -
        pivot_for_delta[("LR2", "full_unpenalized")]
    ).values,
})

print("\n" + "=" * 110)
print("CORRECTED ROMANIA DELTA TABLE")
print("=" * 110)
print(delta)

# ---------------------------------------------------------
# 7. save corrected final outputs
# ---------------------------------------------------------
save_final_long = output_dir / "RO_final_summary_long_corrected.csv"
save_final_wide = output_dir / "RO_final_summary_wide_corrected.csv"
save_delta = output_dir / "RO_final_penalized_minus_unpenalized_delta_corrected.csv"

final_long.to_csv(save_final_long, index=False)
final_wide.to_csv(save_final_wide, index=False)
delta.to_csv(save_delta, index=False)

print("\nSaved corrected files to:")
print(save_scad_long)
print(save_scad_wide)
print(save_scad_audit)
print(save_final_long)
print(save_final_wide)
print(save_delta)

Loaded:
/content/drive/MyDrive/romania_results/RO_2007_2009_unpenalized_same_sample_metrics_combined.csv
/content/drive/MyDrive/romania_results/RO_2007_2009_scad_bic_same_sample_metrics_combined.csv

AUDIT TABLE: ORIGINAL VS RECOMPUTED SCAD-BIC LR2
   dataset  year              model  Gini_total   Gi_expl  LR2_pkg_original  \
0  RO_2007  2007  extended_scad_bic    0.398588  0.138480          0.334935   
1  RO_2007  2007      full_scad_bic    0.398588  0.165407          0.400062   
2  RO_2009  2009  extended_scad_bic    0.360920  0.102861          0.295500   
3  RO_2009  2009      full_scad_bic    0.360920  0.129571          0.372233   

   LR2_recomputed_from_Gi_over_Gini  
0                          0.347427  
1                          0.414983  
2                          0.284997  
3                          0.359003  

CORRECTED ROMANIA FINAL LONG SUMMARY TABLE
   dataset  year                 model  n_used  Gini_total   Gi_expl  \
0  RO_2007  2007  extended_unpenalized   13648   

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA -> SPAIN-ALIGNED THESIS-READY FINAL TABLES
# Make Romania final presentation schema identical to Spain
# =========================================================

input_dir = Path("/content/drive/MyDrive/romania_results")
output_dir = input_dir

# ---------------------------------------------------------
# 1. read Romania corrected final long table
# ---------------------------------------------------------
f_long_corrected = input_dir / "RO_final_summary_long_corrected.csv"
f_audit = input_dir / "RO_2007_2009_scad_bic_LR2_audit.csv"

print("corrected long exists:", f_long_corrected.exists(), "|", f_long_corrected)
print("audit exists:", f_audit.exists(), "|", f_audit)

long_df = pd.read_csv(f_long_corrected)
audit_df = pd.read_csv(f_audit)

# ---------------------------------------------------------
# 2. map Romania internal names -> Spain thesis-ready schema
# ---------------------------------------------------------
def map_method(model_name: str) -> str:
    if "scad_bic" in model_name:
        return "SCAD-BIC LR"
    elif "unpenalized" in model_name:
        return "Unpenalized LR"
    else:
        raise ValueError(f"Unknown model name: {model_name}")

def map_scope(model_name: str) -> str:
    if model_name.startswith("extended"):
        return "Extended"
    elif model_name.startswith("full"):
        return "Full"
    else:
        raise ValueError(f"Unknown model scope in: {model_name}")

thesis_long = long_df.copy()
thesis_long["country"] = "RO"
thesis_long["method"] = thesis_long["model"].apply(map_method)
thesis_long["model_scope"] = thesis_long["model"].apply(map_scope)
thesis_long["LR2_final"] = thesis_long["LR2"]

# default: unpenalized package LR2 equals final LR2
thesis_long["LR2_package"] = thesis_long["LR2_final"]
thesis_long["LR2_diff_package_minus_final"] = 0.0

# ---------------------------------------------------------
# 3. merge package LR2 for SCAD-BIC rows from audit table
# ---------------------------------------------------------
audit_use = audit_df[[
    "dataset", "year", "model", "LR2_pkg_original", "LR2_recomputed_from_Gi_over_Gini"
]].copy()

audit_use = audit_use.rename(columns={
    "LR2_pkg_original": "LR2_package_from_audit",
    "LR2_recomputed_from_Gi_over_Gini": "LR2_final_from_audit"
})

thesis_long = thesis_long.merge(
    audit_use,
    on=["dataset", "year", "model"],
    how="left"
)

scad_mask = thesis_long["method"] == "SCAD-BIC LR"

thesis_long.loc[scad_mask, "LR2_package"] = thesis_long.loc[scad_mask, "LR2_package_from_audit"]
thesis_long.loc[scad_mask, "LR2_final"] = thesis_long.loc[scad_mask, "LR2_final_from_audit"]

thesis_long["LR2_diff_package_minus_final"] = (
    thesis_long["LR2_package"] - thesis_long["LR2_final"]
)

# ---------------------------------------------------------
# 4. final column selection + ordering
# ---------------------------------------------------------
thesis_long = thesis_long[[
    "country",
    "year",
    "method",
    "model_scope",
    "n_used",
    "n_selected",
    "Gini_total",
    "Gi_expl",
    "LR2_final",
    "LR2_package",
    "LR2_diff_package_minus_final"
]].copy()

method_order = {
    "SCAD-BIC LR": 1,
    "Unpenalized LR": 2
}
scope_order = {
    "Extended": 1,
    "Full": 2
}

thesis_long["method_order"] = thesis_long["method"].map(method_order)
thesis_long["scope_order"] = thesis_long["model_scope"].map(scope_order)

thesis_long = thesis_long.sort_values(
    ["year", "method_order", "scope_order"]
).reset_index(drop=True)

thesis_long = thesis_long.drop(columns=["method_order", "scope_order"])

# round to Spain-style display precision
for col in ["Gini_total", "Gi_expl", "LR2_final", "LR2_package", "LR2_diff_package_minus_final"]:
    thesis_long[col] = thesis_long[col].round(4)

# keep n_selected as blank for unpenalized rows if preferred
# thesis_long.loc[thesis_long["method"] == "Unpenalized LR", "n_selected"] = pd.NA

print("\n" + "=" * 110)
print("ROMANIA FINAL LONG TABLE (SPAIN-ALIGNED, THESIS-READY)")
print("=" * 110)
print(thesis_long)

# ---------------------------------------------------------
# 5. wide table in Spain's style
# index = method + model_scope
# columns = year
# values = n_used, n_selected, Gini_total, Gi_expl, LR2_final, LR2_package
# ---------------------------------------------------------
thesis_wide = thesis_long.pivot(
    index=["method", "model_scope"],
    columns="year",
    values=["n_used", "n_selected", "Gini_total", "Gi_expl", "LR2_final", "LR2_package"]
)

print("\n" + "=" * 110)
print("ROMANIA FINAL WIDE TABLE (2007 vs 2009, SPAIN-ALIGNED)")
print("=" * 110)
print(thesis_wide)

# ---------------------------------------------------------
# 6. rows where package LR2 differs from final LR2
# exactly like Spain's discrepancy check
# ---------------------------------------------------------
discrepancy = thesis_long[
    thesis_long["LR2_diff_package_minus_final"].abs() > 1e-12
].copy()

print("\n" + "=" * 110)
print("ROMANIA ROWS WHERE PACKAGE LR2 DIFFERS FROM FINAL LR2")
print("=" * 110)
print(discrepancy)

# ---------------------------------------------------------
# 7. save
# ---------------------------------------------------------
save_long = output_dir / "RO_final_results_long_unified_lr2.csv"
save_wide = output_dir / "RO_final_results_wide_unified_lr2.csv"
save_disc = output_dir / "RO_lr2_discrepancy_check.csv"

thesis_long.to_csv(save_long, index=False)
thesis_wide.to_csv(save_wide)
discrepancy.to_csv(save_disc, index=False)

print("\nSaved to:")
print(save_long)
print(save_wide)
print(save_disc)

corrected long exists: True | /content/drive/MyDrive/romania_results/RO_final_summary_long_corrected.csv
audit exists: True | /content/drive/MyDrive/romania_results/RO_2007_2009_scad_bic_LR2_audit.csv

ROMANIA FINAL LONG TABLE (SPAIN-ALIGNED, THESIS-READY)
  country  year          method model_scope  n_used  n_selected  Gini_total  \
0      RO  2007     SCAD-BIC LR    Extended   13648        21.0      0.3986   
1      RO  2007     SCAD-BIC LR        Full   13648        38.0      0.3986   
2      RO  2007  Unpenalized LR    Extended   13648         NaN      0.3986   
3      RO  2007  Unpenalized LR        Full   13648         NaN      0.3986   
4      RO  2009     SCAD-BIC LR    Extended   12977        16.0      0.3609   
5      RO  2009     SCAD-BIC LR        Full   12977        41.0      0.3609   
6      RO  2009  Unpenalized LR    Extended   12977         NaN      0.3609   
7      RO  2009  Unpenalized LR        Full   12977         NaN      0.3609   

   Gi_expl  LR2_final  LR2_pack

In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# ROMANIA - FINAL PATCH (ALL-IN-ONE)
# 1) Fix penalized LR2 consistency
# 2) Build Romania final long/wide tables
# 3) Save EVERYTHING into ONE Excel workbook
# =========================================================

base = Path("/content/drive/MyDrive/romania_results")

# ---------------------------------------------------------
# A. Read source files
# ---------------------------------------------------------
f_unpen = base / "RO_2007_2009_unpenalized_same_sample_metrics_combined.csv"
f_scad  = base / "RO_2007_2009_scad_bic_same_sample_metrics_combined.csv"

if not f_unpen.exists():
    raise FileNotFoundError(f"Missing file: {f_unpen}")
if not f_scad.exists():
    raise FileNotFoundError(f"Missing file: {f_scad}")

u = pd.read_csv(f_unpen)
s = pd.read_csv(f_scad)

print("Loaded:")
print(f_unpen)
print(f_scad)

# ---------------------------------------------------------
# B. Audit + recompute SCAD-BIC LR2
# ---------------------------------------------------------
s = s.copy()
s["LR2_package"] = s["LR2"]
s["LR2_final"] = s["Gi_expl"] / s["Gini_total"]
s["LR2_diff_package_minus_final"] = s["LR2_package"] - s["LR2_final"]

audit_df = s[[
    "dataset", "year", "model", "n_used", "n_selected",
    "Gini_total", "Gi_expl",
    "LR2_package", "LR2_final", "LR2_diff_package_minus_final"
]].copy()

print("\n=== SCAD-BIC AUDIT TABLE ===")
print(audit_df)

# Replace LR2 with unified final LR2 for downstream summary
s["LR2"] = s["LR2_final"]

# ---------------------------------------------------------
# C. Prepare unpenalized rows in same schema
# ---------------------------------------------------------
u = u.copy()
if "n_selected" not in u.columns:
    u["n_selected"] = pd.NA

u["LR2_package"] = u["LR2"]
u["LR2_final"] = u["Gi_expl"] / u["Gini_total"]
u["LR2_diff_package_minus_final"] = u["LR2_package"] - u["LR2_final"]

# ---------------------------------------------------------
# D. Combine corrected long table
# ---------------------------------------------------------
combined = pd.concat([u, s], ignore_index=True)

combined["country"] = combined["dataset"].str.split("_").str[0]
combined["year"] = combined["dataset"].str.split("_").str[1].astype(int)

def parse_scope(model: str) -> str:
    if model.startswith("extended"):
        return "Extended"
    elif model.startswith("full"):
        return "Full"
    return model

def parse_method(model: str) -> str:
    if "unpenalized" in model:
        return "Unpenalized LR"
    elif "scad_bic" in model:
        return "SCAD-BIC LR"
    return model

combined["model_scope"] = combined["model"].apply(parse_scope)
combined["method"] = combined["model"].apply(parse_method)

final_long = combined[[
    "country", "year", "method", "model_scope",
    "n_used", "n_selected",
    "Gini_total", "Gi_expl",
    "LR2_final", "LR2_package", "LR2_diff_package_minus_final"
]].copy()

final_long["n_selected"] = final_long["n_selected"].astype("Int64")

final_long = final_long.sort_values(
    ["year", "method", "model_scope"]
).reset_index(drop=True)

# ---------------------------------------------------------
# E. Rounded thesis-ready version
# ---------------------------------------------------------
final_long_round = final_long.copy()
for col in ["Gini_total", "Gi_expl", "LR2_final", "LR2_package", "LR2_diff_package_minus_final"]:
    final_long_round[col] = final_long_round[col].round(4)

print("\n=== FINAL LONG TABLE (THESIS-READY) ===")
print(final_long_round)

# ---------------------------------------------------------
# F. Wide table
# ---------------------------------------------------------
final_wide = final_long_round.pivot(
    index=["method", "model_scope"],
    columns="year",
    values=["n_used", "n_selected", "Gini_total", "Gi_expl", "LR2_final", "LR2_package"]
)

final_wide_flat = final_wide.copy()
final_wide_flat.columns = [f"{a}_{b}" for a, b in final_wide_flat.columns]
final_wide_flat = final_wide_flat.reset_index()

print("\n=== FINAL WIDE TABLE (2007 vs 2009) ===")
print(final_wide_flat)

# ---------------------------------------------------------
# G. Discrepancy rows
# ---------------------------------------------------------
diff_rows = final_long_round[
    final_long_round["LR2_diff_package_minus_final"].abs() > 1e-6
].copy()

print("\n=== ROWS WHERE PACKAGE LR2 DIFFERS FROM FINAL LR2 ===")
print(diff_rows[[
    "year", "method", "model_scope",
    "Gini_total", "Gi_expl", "LR2_final", "LR2_package",
    "LR2_diff_package_minus_final"
]])

# ---------------------------------------------------------
# H. Save EVERYTHING into ONE Excel workbook
# ---------------------------------------------------------
bundle_path = base / "RO_step4_final_reporting_bundle.xlsx"

with pd.ExcelWriter(bundle_path, engine="openpyxl") as writer:
    final_long_round.to_excel(writer, sheet_name="final_long", index=False)
    final_wide_flat.to_excel(writer, sheet_name="final_wide", index=False)
    diff_rows.to_excel(writer, sheet_name="lr2_discrepancy", index=False)
    audit_df.to_excel(writer, sheet_name="scad_bic_audit", index=False)

print("\nSaved bundled file to:")
print(bundle_path)

Loaded:
/content/drive/MyDrive/romania_results/RO_2007_2009_unpenalized_same_sample_metrics_combined.csv
/content/drive/MyDrive/romania_results/RO_2007_2009_scad_bic_same_sample_metrics_combined.csv

=== SCAD-BIC AUDIT TABLE ===
   dataset  year              model  n_used  n_selected  Gini_total   Gi_expl  \
0  RO_2007  2007  extended_scad_bic   13648          21    0.398588  0.138480   
1  RO_2007  2007      full_scad_bic   13648          38    0.398588  0.165407   
2  RO_2009  2009  extended_scad_bic   12977          16    0.360920  0.102861   
3  RO_2009  2009      full_scad_bic   12977          41    0.360920  0.129571   

   LR2_package  LR2_final  LR2_diff_package_minus_final  
0     0.334935   0.347427                     -0.012492  
1     0.400062   0.414983                     -0.014921  
2     0.295500   0.284997                      0.010503  
3     0.372233   0.359003                      0.013230  

=== FINAL LONG TABLE (THESIS-READY) ===
  country  year          method mo

In [ ]:
!pip -q install rpy2
%load_ext rpy2.ipython

In [ ]:
%%R

library(dplyr)

input_dir <- "/content/drive/MyDrive/romania_results"
output_dir <- "/content/drive/MyDrive/romania_results"

files <- list(
  RO_2007 = file.path(input_dir, "RO_2007_model_ready_with_references.csv"),
  RO_2009 = file.path(input_dir, "RO_2009_model_ready_with_references.csv")
)

# Romania final modeling references
# IMPORTANT: DB040 here is the harmonized region label used for modeling
reference_map <- list(
  PB150 = "1",
  PB220A = "RO",
  DB040 = "Nord_Est",
  age_group = "25_54",
  PE040 = "1",
  PH010 = "2",
  economic_status = "1",
  PL040 = "3",
  PL050 = "9",
  HH030 = "6",
  HS040 = "1"
)

cat_vars <- c(
  "PB150", "PB220A", "DB040", "age_group",
  "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
)

normalize_code <- function(x) {
  y <- as.character(x)
  y <- trimws(y)
  y <- sub("\\.0$", "", y)
  y
}

levels_with_ref_first <- function(x, ref) {
  lev <- sort(unique(x[!is.na(x)]))
  if (!(ref %in% lev)) stop(paste("Reference", ref, "not found"))
  c(ref, lev[lev != ref])
}

extract_lm_table <- function(fit, dataset, year, model_scope) {
  s <- summary(fit)$coefficients
  data.frame(
    dataset = dataset,
    year = year,
    method = "Linear regression",
    model_scope = model_scope,
    term = rownames(s),
    estimate = s[, 1],
    std_error = s[, 2],
    t_value = s[, 3],
    p_value = s[, 4],
    row.names = NULL
  )
}

metrics_list <- list()
coef_list <- list()

for (dataset_name in names(files)) {
  cat("\n", strrep("=", 90), "\n", sep = "")
  cat("RUNNING:", dataset_name, "\n")
  cat(strrep("=", 90), "\n", sep = "")

  d <- read.csv(files[[dataset_name]], stringsAsFactors = FALSE)

  d <- d[, c(
    "eq_income", "PB040",
    "PB150", "PB220A", "DB040", "age_group",
    "PE040", "PH010", "economic_status", "PL040", "PL050", "HH030", "HS040"
  )]

  for (v in cat_vars) {
    d[[v]] <- normalize_code(d[[v]])
  }

  d$eq_income <- as.numeric(d$eq_income)
  d$PB040 <- as.numeric(d$PB040)

  # references
  d$PB150 <- factor(d$PB150, levels = levels_with_ref_first(d$PB150, reference_map$PB150))
  d$PB220A <- factor(d$PB220A, levels = levels_with_ref_first(d$PB220A, reference_map$PB220A))
  d$DB040 <- factor(d$DB040, levels = levels_with_ref_first(d$DB040, reference_map$DB040))
  d$age_group <- factor(d$age_group, levels = c("25_54", "15_24", "55_64", "65_plus"))
  d$PE040 <- factor(d$PE040, levels = levels_with_ref_first(d$PE040, reference_map$PE040))
  d$PH010 <- factor(d$PH010, levels = levels_with_ref_first(d$PH010, reference_map$PH010))
  d$economic_status <- factor(d$economic_status, levels = levels_with_ref_first(d$economic_status, reference_map$economic_status))
  d$PL040 <- factor(d$PL040, levels = levels_with_ref_first(d$PL040, reference_map$PL040))
  d$PL050 <- factor(d$PL050, levels = levels_with_ref_first(d$PL050, reference_map$PL050))
  d$HH030 <- factor(d$HH030, levels = levels_with_ref_first(d$HH030, reference_map$HH030))
  d$HS040 <- factor(d$HS040, levels = levels_with_ref_first(d$HS040, reference_map$HS040))

  d$w_norm <- d$PB040 / sum(d$PB040)
  d$log_eq_income <- log(d$eq_income)

  year_val <- as.integer(sub("RO_", "", dataset_name))

  cat("n =", nrow(d), "\n")
  cat("Missing counts:\n")
  print(colSums(is.na(d)))

  cat("\nObserved non-empty levels after factor conversion:\n")
  print(sapply(cat_vars, function(v) nlevels(droplevels(d[[v]]))))

  fit_ext <- lm(
    log_eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010,
    data = d,
    weights = w_norm
  )

  fit_full <- lm(
    log_eq_income ~ PB150 + PB220A + DB040 + age_group + PE040 + PH010 +
      economic_status + PL040 + PL050 + HH030 + HS040,
    data = d,
    weights = w_norm
  )

  s_ext <- summary(fit_ext)
  s_full <- summary(fit_full)

  metrics_list[[dataset_name]] <- data.frame(
    dataset = dataset_name,
    year = year_val,
    method = "Linear regression",
    model_scope = c("Extended", "Full"),
    n_used = c(nrow(d), nrow(d)),
    outcome = c("log_eq_income", "log_eq_income"),
    r_squared = c(s_ext$r.squared, s_full$r.squared),
    adj_r_squared = c(s_ext$adj.r.squared, s_full$adj.r.squared),
    aic = c(AIC(fit_ext), AIC(fit_full)),
    bic = c(BIC(fit_ext), BIC(fit_full))
  )

  coef_ext <- extract_lm_table(fit_ext, dataset_name, year_val, "Extended")
  coef_full <- extract_lm_table(fit_full, dataset_name, year_val, "Full")
  coef_list[[dataset_name]] <- bind_rows(coef_ext, coef_full)

  cat("\nExtended model done.\n")
  cat("Full model done.\n")
  cat("Extended R2 =", s_ext$r.squared, "\n")
  cat("Full R2 =", s_full$r.squared, "\n")
}

metrics_all <- bind_rows(metrics_list)
coef_all <- bind_rows(coef_list)

write.csv(
  metrics_all,
  file.path(output_dir, "RO_2007_2009_linear_regression_metrics.csv"),
  row.names = FALSE
)

write.csv(
  coef_all,
  file.path(output_dir, "RO_2007_2009_linear_regression_coefficients.csv"),
  row.names = FALSE
)

cat("\nSaved metrics to:\n")
cat(file.path(output_dir, "RO_2007_2009_linear_regression_metrics.csv"), "\n")

cat("\nSaved coefficients to:\n")
cat(file.path(output_dir, "RO_2007_2009_linear_regression_coefficients.csv"), "\n")


RUNNING: RO_2007 
n = 13648 
Missing counts:
      eq_income           PB040           PB150          PB220A           DB040 
              0               0               0               0               0 
      age_group           PE040           PH010 economic_status           PL040 
              0               0               0               0               0 
          PL050           HH030           HS040          w_norm   log_eq_income 
              0               0               0               0               0 

Observed non-empty levels after factor conversion:
          PB150          PB220A           DB040       age_group           PE040 
              2               3               8               4               6 
          PH010 economic_status           PL040           PL050           HH030 
              5               8               4               9               6 
          HS040 
              2 

Extended model done.
Full model done.
Extended R2 = 0.068


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

